### 05장 판다스 집계와 계산  
이번 장에서는 주문 데이터와 고객 데이터를 이용해 수치형 컬럼을 계산하고, 범주별로 묶어 집계하며, 두 데이터프레임을 병합하는 방법을 학습한다. 같은 데이터는 장 앞부분에서 한 번만 불러오고 뒤의 항목에서는 orders와 customers 데이터프레임을 이어서 사용한다.  

In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('./data/pandas_orders.csv')
customers = pd.read_csv('./data/pandas_customers.csv')

print(orders.shape)
print(customers.shape)

(200, 10)
(60, 4)


orders는 주문 한 건을 한 행으로 가지는 100행 10열의 데이터프레임이다.  
customrs는 고객 한 명을 한 행으로 가지는 60행 4열의 데이터프레임이다.  

#### 02. 기초 통계 함수  
기초 통계 함수는 데이터의 전체 규모와 분포를 빠르게 확인하는 계산 도구이다.  
작업형 1에서는 조건에 맞는 데이터의 합계, 평균, 중앙값, 최댓값, 최솟값을 구하는 문제가 자주 나오고, 작업형 2에서는 모델링 전에 수치형 컬럼의 범위를 점검할 때 사용한다.  
평균이나 합계를 구하기 전에 계산할 컬럼이 수치형인지 확인하고, 문제에서 요구한 컬럼만 선택하면 코드가 단순해진다.

In [2]:
cols = ['order_id', 'qty', 'unit_price', 'discount_rate', 'sales_amount']
print(orders[cols].head(6).to_string(index=False))
print(orders[['qty', 'unit_price', 'discount_rate', 'sales_amount']].dtypes)

order_id  qty  unit_price  discount_rate  sales_amount
  O00001   11       29100           0.05        304095
  O00002   11       91700           0.00       1008700
  O00003    2       60300           0.03        116982
  O00004    5       60500           0.00        302500
  O00005    5       10000           0.00         50000
  O00006    2       98700           0.00        197400
qty                int64
unit_price         int64
discount_rate    float64
sales_amount       int64
dtype: object


#### 합계, 평균, 중앙값과 범위  
가장 기본이 되는 통계 함수부터 하나씩 적용해본다. sum(), mean(), median(), min(), max()는 모두 시리즈에 바로 붙여 사용할 수 있다.

In [3]:
print(orders['sales_amount'].sum())
print(orders['sales_amount'].mean())
print(orders['sales_amount'].median())
print(orders['sales_amount'].min())
print(orders['sales_amount'].max())

75020932
375104.66
277067.5
12200
1313400


#### 여러 통계량 한 번에 확인  
1. agg()로 여러 함수의 통계량 확인  
통계 함수를 하나씩 호출하면 코드가 길어진다. agg()에 함수 이름을 리스트로 전달하면 여러 컬럼의 여러 통계량을 한 표로 확인할 수 있다.

In [4]:
summary = orders[['qty', 'unit_price', 'sales_amount']].agg(['min', 'max', 'median', 'mean']).round(2)
print(summary.to_string())

          qty  unit_price  sales_amount
min      1.00      4900.0      12200.00
max     11.00    119400.0    1313400.00
median   6.00     63150.0     277067.50
mean     6.06     62120.5     375104.66


2. describe()로 전체 분포 확인  
필요한 통계량을 일일이 지정하지 않고 분포 전체를 훑어보고 싶다면 describe()를 활용

In [5]:
desc = orders[['qty', 'unit_price', 'sales_amount']].describe().round(2)
print(desc.to_string())

          qty  unit_price  sales_amount
count  200.00      200.00        200.00
mean     6.06    62120.50     375104.66
std      3.18    33263.53     317950.73
min      1.00     4900.00      12200.00
25%      3.00    34525.00     113252.00
50%      6.00    63150.00     277067.50
75%      9.00    92000.00     571125.00
max     11.00   119400.00    1313400.00


describe()는 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 한 번에 보여준다! 50% 값은 중앙값과 같고, 25%와 75%는 각각 1사분위수와 3사분위수이다.

agg()와 describe()의 선택  
문제에서 필요한 통계량이 정해져 있으면 agg()가 깔끔해진다. 반대로 데이터의 전반적인 상태를 빠르게 살펴볼 때는 describe()가 편하다.  
시험 답안은 필요한 값만 정확히 출력해야 하므로, 최종 계산에서는 개별 함수나 agg()로 결과를 좁히는 것이 좋다

#### 조건을 적용한 계산  
통계량은 계산으로 끝나지 않고 조건과 결합되는 경우가 많다. 평균 이상인 주문이 몇 건이지 세려면 먼저 평균을 구한 뒤 이를 조건식에  
사용한다.

In [6]:
mean_sales = orders['sales_amount'].mean()
cond = orders['sales_amount'] >= mean_sales

print(mean_sales)
print(cond.sum())

375104.66
75


#### 03. value_counts  
value_counts()는 한 컬럼에서 각 값이 몇 번씩 나타나는지 세는 함수이다. 회원 등급, 가입 채널, 지역, 상품 분류처럼   
같은 값이 반복되는 범주형 컬럼에 적합하다.  
작업형 1에서는 특정 범주의 개수나 가장 많이 등장한 값을 묻는 문제가 자주 나오며, 작업형 2에서는 범주형 컬럼의 분포를 한쪽으로 치우쳐졌는지 확인할 때 사용할 수 있다.

In [7]:
# 기본 빈도 계산
print(customers['member_grade'].value_counts().to_string())
print(customers['join_channel'].value_counts().to_string())

member_grade
basic     26
silver    21
vip        7
gold       6
join_channel
app        26
web        21
offline    13


#### 인덱스 순서와 비율  
기본 정렬은 개수 기준이므로 10대, 20대 같은 순서가 뒤섞일 수 있다. 이럴 때는 sort_index()로 순서를 정리하고, 비율이 필요하면  
normalize=True를 사용한다.

In [8]:
age_count = customers['age_group'].value_counts().sort_index()  
ratio = customers['join_channel'].value_counts(normalize=True).round(3) # normalize=True는 정규화를 해주는 것

print(age_count.to_string())
print(ratio.to_string())

age_group
10대        4
20대       19
30대       21
40대       12
50대 이상     4
join_channel
app        0.433
web        0.350
offline    0.217


#### 조건을 적용한 빈도
전체가 아니라 특정 조건에 해당하는 데이터 안에서의 분포를 보고 싶을 때도 있음. app으로 가입한 고객만 골라낸 뒤 회원등급 빈도 계산

In [9]:
app_customers = customers[customers['join_channel'] == 'app'] # 가입 채널이 'app'인 애들의 member_grade 가져오는 코드
print(app_customers['member_grade'].value_counts().to_string())

member_grade
silver    10
basic      9
vip        4
gold       3


#### 4. 두 컬럼의 조합 빈도  
한 컬럼씩 따로 보는 대신 두 컬럼의 조합이 몇 번 나타나는지 셀 수 없다

In [10]:
comb = customers[['member_grade', 'join_channel']].value_counts()
print(comb.head(8).to_string()) # 이때 내림차순으로 나타남

member_grade  join_channel
silver        app             10
basic         app              9
              offline          9
              web              8
silver        web              8
vip           app              4
gold          app              3
              web              3


빈도 결과를 표로 바꾸기  
value_counts() 결과는 시리즈 형태이다. 결과를 표처럼 다루고 싶다면, reset_index()를 사용할 수 있다. 표로 바꾸면 컬럼명을 붙이거나, 빈도 컬럼을 기준으로 정렬하고 조건을 적용하기 편하다.

#### groupby  
groupby()는 데이터를 특정 기준으로 나눈 뒤 각 그룹에 합계, 평균, 개수와 같은 계산을 수행할 때 사용한다.  
주문 데이터를 지역별롤 나누어 매출 합계를 구하거나 상품 분류별 평균 주문 수량을 구할 때 필요하다.

groupby의 기본 구조는 orders.groupby('기준 컬럼')['계산할 컬럼'].집계함수()  
예를 들어, 지역별 매출 합계는 orders.groupby('region')['sales_amount'].sum()으로 계산함

In [11]:
# groupby
# 그룹별 합계
region_sales = orders.groupby('region')['sales_amount'].sum()
print(region_sales.sort_values(ascending=False).to_string())

region
강원권    15230717
제주권    13563540
호남권    13529997
영남권    13118465
수도권    12078330
충청권     7499883


위 코드에서 region_sales = orders.groupby('region')['sales_amount'].sum() 이 부분은 지역이 같은 행끼리 묶고, 지역별 'sales_amount'의 합계를 각 지역별로 나타냄

#### 그룹별 행 수와 평균  
합계 외에도 그룹별 주문 건수나 평균이 필요한 경우가 많다. 행 수는 size(), 평균은 mean()으로 계산한다.  

In [12]:
channel_count = orders.groupby('channel').size()
category_qty = orders.groupby('category')['qty'].mean().round(2)

print(channel_count.sort_values(ascending=False).to_string(), end="\n\n")
print(category_qty.to_string())

channel
app      80
web      53
store    50
phone    17

category
beauty     5.17
book       5.97
digital    5.94
kitchen    6.90
office     6.15
sports     6.34


#### 여러 기준으로 묶기  
기준이 하나일 필요는 없음. groupby()에 컬럼을 리스트로 전달하면 두 기준의 조합별로 묶을 수 있음

In [13]:
combo_sales = orders.groupby(['region', 'category'])['sales_amount'].sum()
print(combo_sales.sort_values(ascending=False).head(8).to_string())

region  category
호남권     office      4584400
영남권     kitchen     4140623
수도권     book        3584369
호남권     sports      3423940
수도권     sports      3371279
강원권     digital     3290272
        office      3268410
        kitchen     2914100


#### 4. 결과를 데이터프레임으로 만들기  
groupby() 결과는 기준 칼럼이 인덱스로 들어간 시리즈 형태라 후속 작업에 번거로울 수 있다.  
as_index=False를 사용하면 처음부터 데이터프레임 형태로 결과를 받을 수 있다.

In [14]:
region_table = orders.groupby('region', as_index=False)['sales_amount'].sum() # as_index=False 덕분에 데이터프레임 형태임
region_table = region_table.sort_values('sales_amount', ascending=False)
print(region_table.to_string(index=False))

region  sales_amount
   강원권      15230717
   제주권      13563540
   호남권      13529997
   영남권      13118465
   수도권      12078330
   충청권       7499883


groupby 결과에서 값 하나 꺼내기  
작업형 1에서는 전체 집계표가 아니라 특정 값 하나만 답으로 제출하는 경우가 많다. 지역별 매출 합계에서 호남권 값만 필요하다면 groupby 결과에 loc['호남권']을 사용하면 된다

#### 5. agg  
agg()는 aggregation의 줄임말로 여러 집계 함수를 한 번에 적용할 때 사용한다. groupby()로 기준별 묶음을 만든 뒤 합계, 평균, 최대값, 개수 등을 한 표에서 비교할 수 있다.  
agg()는 이 칼럼에 이 계산들을 적용해 달라고 지정하는 도구이다. 주문금액의 합계, 평균, 최댓값을 각각 따로 계산하지 않고 agg(['sum', 'mean', 'max'])처럼 구할 수 있다.

In [15]:
# 한 컬럼에 여러 함수 적용  
# 주문금액과 주문 수량 각각에 필요한 함수 목록을 전달해 여러 통계량을 한 번에 계산한다.
sales_summary = orders['sales_amount'].agg(['sum', 'mean', 'max'])
qty_summary = orders['qty'].agg(['min', 'max', 'mean', 'sum'])

print(sales_summary.round(2).to_string())
print(qty_summary.round(2).to_string())

sum     75020932.00
mean      375104.66
max      1313400.00
min        1.00
max       11.00
mean       6.06
sum     1211.00


groupby와 agg의 결합  
groupby() 뒤에 agg()를 이어붙이면 그룹별로 여러 통계량을 한 표에 담을 수 있다. 채널별 주문 건수, 합계, 평균, 최댓값을 함께 계산해보자!

In [16]:
channel_summary = orders.groupby('channel')['sales_amount'].agg(['count', 'sum', 'mean', 'max']).round(1)
print(channel_summary.to_string())

         count       sum      mean      max
channel                                    
app         80  32723394  409042.4  1313400
phone       17   5312283  312487.2   951900
store       50  16906597  338131.9  1062100
web         53  20078658  378842.6  1248390


In [17]:
# 여러 컬럼에 서로 다른 함수 적용  
category_summary = orders.groupby('category').agg({'qty': ['sum', 'mean'], 'sales_amount': ['sum', 'mean']}).round(2)
print(category_summary.to_string())

          qty       sales_amount           
          sum  mean          sum       mean
category                                   
beauty    186  5.17      9397217  261033.81
book      197  5.97     13201466  400044.42
digital   184  5.94     10560525  340662.10
kitchen   214  6.90     13917022  448936.19
office    246  6.15     15394031  384850.78
sports    184  6.34     12550671  432781.76


딕셔너리의 key에는 계산할 컬럼을, value에는 적용할 집계 함수 목록을 지정한다. 여러 칼럼을 동시에 집계하면 결과 컬럼이 여러 단계로 표시된다.

이름을 붙인 집계  
집계 결과의 컬럼명이 sum, mean처럼 함수 이름 그대로면 의미를 알아보기 어렵다.  
이름을 붙인 집계를 사용하면 결과 컬럼명을 직접 정할 수 있다.

In [18]:
region_summary = orders.groupby('region').agg(order_counts=('order_id', 'count'), 
                                              total_qty=('qty', 'sum'), 
                                              total_sales=('sales_amount', 'sum'), 
                                              avg_sales=('sales_amount', 'mean')).round(1)
print(region_summary.sort_values('total_sales', ascending=False).to_string())

        order_counts  total_qty  total_sales  avg_sales
region                                                 
강원권               38        246     15230717   400808.3
제주권               30        191     13563540   452118.0
호남권               35        209     13529997   386571.3
영남권               38        218     13118465   345222.8
수도권               35        220     12078330   345095.1
충청권               24        127      7499883   312495.1


이렇게 order_counts나 total_qty로 이름을 정해서 결과 컬럼며을 직접 지정할 수 있다.  
이를 이용하면 정렬이나, 특정 컬럼 선택을 이어서 적용하기 편하다.

#### pivot_table  
pivot_table()은 데이터를 행 기준과 열 기준으로 나누어 요약표를 만들 때 사용한다. groupby()가 기준별 계산 결과를 행 중심으로 보여 준다면, pivot_table()은 행과 열을 교차시켜서 표 형태로 정리하는 데 적합하다

pivot_table의 기본 구조  
pd.pivot_table(데이터, index='행 기준', columns='열 기준', values='계산할 칼럼', aggfunc='sum')
columns가 필요하지 않은 문제에서는 해당 옵션을 생략할 수 있다

한 개 기준으로 피벗 테이블 만들기  
가장 단순한 형태부터 만들어보자! 기준 하나에 계산할 컬럼 하나를 지정하면 groupby()와 같은 결과를 표 형태로 얻을 수 있다

In [19]:
region_sales = pd.pivot_table(
    orders,
    values='sales_amount', # 계산할 열
    index='region', # 행 기준
    aggfunc='sum'
)
print(region_sales.to_string())

        sales_amount
region              
강원권         15230717
수도권         12078330
영남권         13118465
제주권         13563540
충청권          7499883
호남권         13529997


In [20]:
# 행과 열에 기준 배치
regional_channel = pd.pivot_table(
    orders,
    values='sales_amount',
    index='region',
    columns='channel', # 열 기준
    aggfunc='sum',
    fill_value=0
)

print(regional_channel.to_string())

channel      app    phone    store      web
region                                     
강원권      5900832  2083900  2661390  4584595
수도권      5376029   420700  3198115  3083486
영남권      3411744   807080  2970500  5929141
제주권      7998749  1137392  3170673  1256726
충청권      2479303   279216  2130754  2610610
호남권      7556737   583995  2775165  2614100


#### 평균과 여러 값 요약  
기본 계산  
합계 대신 평균이 필요하면 aggfunc만 바꾸면 된다. 상품 분류와 채널 조합별 평균 주문 수량을 계산해보자!

In [21]:
qty_pivot = pd.pivot_table(
    orders,
    values='qty', # 계산할 열
    index='category', # 행 기준
    columns='channel', # 열 기준
    aggfunc='mean'
).round(2)
print(qty_pivot.to_string())

channel    app  phone  store   web
category                          
beauty    5.25   6.00   5.50  4.12
book      5.31    NaN   6.25  6.89
digital   5.93  11.00   4.71  6.38
kitchen   7.55   6.75   6.00  6.89
office    7.53   6.80   4.80  5.10
sports    5.82   9.67   5.83  6.22


In [22]:
# 여러 값에 서로 다른 계산 적용
category_summary = pd.pivot_table(
    orders,
    values=['qty', 'sales_amount'],
    index='category',
    aggfunc={'qty': 'sum', 'sales_amount':'mean'}
).round(1)
print(category_summary.to_string())

          qty  sales_amount
category                   
beauty    186      261033.8
book      197      400044.4
digital   184      340662.1
kitchen   214      448936.2
office    246      384850.8
sports    184      432781.8


In [24]:
# 전체 합계 표시  
sales_pivot = pd.pivot_table(
    orders,
    values='sales_amount',
    index='region',
    columns='channel',
    aggfunc='sum',
    fill_value=0,
    margins=True # 전체 합계를 보기 위해 margins를 True로 지정, 즉, all 행과 all 열이 추가됨
)
print(sales_pivot.to_string())

channel       app    phone     store       web       All
region                                                  
강원권       5900832  2083900   2661390   4584595  15230717
수도권       5376029   420700   3198115   3083486  12078330
영남권       3411744   807080   2970500   5929141  13118465
제주권       7998749  1137392   3170673   1256726  13563540
충청권       2479303   279216   2130754   2610610   7499883
호남권       7556737   583995   2775165   2614100  13529997
All      32723394  5312283  16906597  20078658  75020932


특정 교차 지점의 값은 loc 이용해서 꺼내기 가능!
sales_pivot.loc['수도권', 'store']

#### 7. apply와 map  
map()과 apply()는 기존 컬럼의 값을 다른 값으로 변환하거나 새로운 기준으로 분류할때 사용한다.  
map()은 대응표가 명확한 값 변환에 간단하고, apply()는 직접 만든 계산 규칙이나 조건을 적용할때 유용하다.  
즉, map()은 A는 B로, C는 D로 바꾼다는 대응표가 있을 때 편하다. apply는 이 값이 0보다 크면 할인, 아니면 정가처럼 조건이나 규칙을 직접 적용할 때 편하다.

map으로 값 매핑  
map()은 딕셔너리로 만든 대응표에 따라 값을 다른 값으로 바꾸는 함수이다. 영문 채널명을 한글로 바꾸는 대응표를 만들어 적용해보자!

In [26]:
# 원본 orders를 유지하기 위해 복사본 사용
orders_work = orders.copy()

channel_name = {
    'app' : '앱',
    'web' : '웹',
    'store' : '매장',
    'phone' : '전화'
}

orders_work['channel_name'] = orders_work['channel'].map(channel_name)
print(orders_work[['channel', 'channel_name']].head(8).to_string())

  channel channel_name
0   phone           전화
1     web            웹
2     app            앱
3     web            웹
4     web            웹
5     app            앱
6     web            웹
7     web            웹


In [27]:
# 상품 분류 이름 변환
category_name = {
    'digital' : '디지털', 'kitchen' : '주방', 'office' : '사무', 'beauty' : '뷰티', 'book' : '책', 'sports' : '스포츠'
}
orders_work['category_ko'] = orders_work['category'].map(category_name)
print(orders_work[['category', 'category_ko']].head(6).to_string(index=False))

category category_ko
 digital         디지털
 kitchen          주방
 digital         디지털
 kitchen          주방
  office          사무
  sports         스포츠


#### apply로 조건 적용  
map()이 정해진 대응표로 값을 바꾼다면, apply()는 함수를 적용해 값을 만들어 낸다. 할인율이 0보다 크면 '할인', 아니면 '정가'로 구분하는 조건을 lambda 함수로 적용해 본다.

In [28]:
orders_work['discount_flag'] = orders_work['discount_rate'].apply(lambda x: '할인' if x > 0 else '정가')
print(orders_work['discount_flag'].value_counts().to_string())

discount_flag
할인    114
정가     86


In [29]:
# 함수를 만들어 여러 조건 처리  
def price_level(price):
    if price >= 700000:
        return '고액'
    elif price >= 200000:
        return '중간'
    else:
        return '소액'
    
orders_work['sales_level'] = orders_work['sales_amount'].apply(price_level)
print(orders_work['sales_level'].value_counts().to_string())

sales_level
중간    88
소액    78
고액    34


apply를 꼭 써야하는 순간  
한 컬럼의 값을 단순히 대응표로 바꾸는 작업은 map()d이 더 간단하다. 여러 컬럼의 값을 함께 계산하거나 조건이 여러 단계로 나뉘는 때는 apply()가 자연스럽다. 단순 산술 연산은 orders['qty'] * orders['unit_price']처럼 벡터 연산으로 처리할 수도 있다.

#### 8. 데이터프레임 병합  
분석에 필요한 정보가 여러 파일에 나뉘어 있을 때 공통 기준 컬럼을 이용해 데이터프레임을 병합한다. 주문 데이터에는 customer_id와 주문 정보가 있고, 고객 데이터에는 같은 customer_id에 해당하는 연령대와 회원등급이 있다.  
판다스의 merge()는 두 데이터의 같은 기준값을 찾아 정보를 옆에 붙이는 함수이다. 단순히 행을 위아래로 이어 붙이는 작업과는 다르다.

병합 기준 확인  
병합하기 전에 두 데이터으의 크기와 컬럼의 구성을 확인해야 함. 어떤 컬럼을 기준으로 연결할지 눈으로 확인해야 함

In [30]:
print(orders.shape)
print(customers.shape)
print('orders 컬럼 :', list(orders.columns))
print('customers 컬럼 :', list(customers.columns))

(200, 10)
(60, 4)
orders 컬럼 : ['order_id', 'customer_id', 'region', 'channel', 'category', 'qty', 'unit_price', 'discount_rate', 'order_date', 'sales_amount']
customers 컬럼 : ['customer_id', 'age_group', 'member_grade', 'join_channel']


left 병합  
customer_id를 기준으로 orders에 고객 정보를 붙여본다. 주문 데이터를 기준으로 삼을 것이므로 how='left'로 지정

In [31]:
merged = pd.merge(
    orders,
    customers,
    on='customer_id',
    how='left'
)

print(merged.shape)
print(merged[['order_id', 'customer_id', 'region', 'member_grade', 'age_group', 'sales_amount']].head(8).to_string(index=False))

(200, 13)
order_id customer_id region member_grade age_group  sales_amount
  O00001       C0035    호남권        basic       40대        304095
  O00002       C0001    호남권       silver       20대       1008700
  O00003       C0010    충청권        basic       30대        116982
  O00004       C0024    강원권          vip       40대        302500
  O00005       C0045    호남권        basic       10대         50000
  O00006       C0035    수도권        basic       40대        197400
  O00007       C0058    제주권        basic       20대         48060
  O00008       C0033    수도권          vip       30대         53900


In [33]:
# 병합 결과 점검  
print(merged.shape)
print(merged['member_grade'].isna().sum())
print(merged['member_grade'].value_counts().to_string())

(200, 13)
0
member_grade
silver    78
basic     77
vip       29
gold      16


In [35]:
# left와 inner 비교  
left_merge = orders.merge(customers, on='customer_id', how='left')
inner_merge = orders.merge(customers, on='customer_id', how='inner')

print('left merge :', left_merge.shape)
print('inner merge : ', inner_merge.shape)

left merge : (200, 13)
inner merge :  (200, 13)


#### 상관계수  
상관계수는 두 수치형 컬럼의 선형 관계를 -1부터 1 사이의 값으로 나타낸다. 1에 가까우면 두 값이 같은 방향으로 움직이는 경향이 강하고, -1에 가까우면 반대 방향으로 움직이는 경향이 강하다. 0에 가까우면 선형 관계가 약하다고 본다.  
같이 움직이는 정도 보기  
상관계수는 둘이 얼마나 같이 움직이는지를 보는 숫자이다. 양수이면 한쪽이 커질 때 다른 쪽도 커지는 경향이 있고, 음수이면 반대로 움직이는 경향이 있다. 상관계수가 높다고 해서 한 컬럼이 다른 컬럼의 원인이라고 단정할 수는 없다.  
상관 계수 $\neq$ 인과 관계

In [36]:
# 숫자형 컬럼 선택
num = orders[['qty', 'unit_price', 'discount_rate', 'sales_amount']]
print(num.head(6).to_string(index=False))

 qty  unit_price  discount_rate  sales_amount
  11       29100           0.05        304095
  11       91700           0.00       1008700
   2       60300           0.03        116982
   5       60500           0.00        302500
   5       10000           0.00         50000
   2       98700           0.00        197400


#### corr()로 상관계수 표 만들기  
선택한 데이터프레임에 corr()를 적용하면 모든 컬럼 쌍의 상관계수를 표로 얻을 수 있다

In [37]:
corr = num.corr()
print(corr.round(3).to_string())

                 qty  unit_price  discount_rate  sales_amount
qty            1.000       0.135          0.012         0.696
unit_price     0.135       1.000          0.043         0.730
discount_rate  0.012       0.043          1.000        -0.039
sales_amount   0.696       0.730         -0.039         1.000


전체 표가 필요 없는 경우에는 두 컬럼 사이의 관계만 궁금하다면 시리즈끼리 직접 corr() 함수를 사용하면된다

In [38]:
print(orders['qty'].corr(orders['sales_amount']))
print(orders['unit_price'].corr(orders['sales_amount']))
print(orders['discount_rate'].corr(orders['sales_amount']))

0.6956149628784886
0.7300609337943269
-0.039132531197717396


기준 컬럼 중심으로 정렬  
문제에서는 특정 컬럼과 가장 관련이 큰 컬럼을 묻는 경우가 많음. 
상관계수 표에서 sales_amount 열만 꺼낸 뒤 자기 자신을 제외하고 절댓값 기준으로 정렬해보자

In [40]:
sales_corr = corr['sales_amount'].drop('sales_amount')

print(sales_corr.round(3).to_string())
print(sales_corr.abs().sort_values(ascending=False).round(3).to_string())

qty              0.696
unit_price       0.730
discount_rate   -0.039
unit_price       0.730
qty              0.696
discount_rate    0.039


위처럼 절댓값 기준으로 정렬하면 양의 관계와 음의 관계를 모두 포함해 관련 정도를 비교할 수 있다.  
주문금액과의 상관계수 절댓값이 가장 큰 컬럼은 unit_price이다.

#### 분위수 계산  
분위수는 데이터를 작은 값부터 큰 값까지 정렬했을 때 특정 위치에 해당하는 값이다.  
판다스에서는 quantile()을 사용하며, 0.25는 1사분위수, 0.50은 중앙값, 0.75는 3사분위수를 뜻한다.  
작업형 1에서는 특정 분위수보다 크거나 작은 데이터를 골라 계산하는 문제가 나올 수 있으므로 분위수 계산과 조건 필터링을 함께 익히는 것이 좋다!

1. 한개의 분위수 계산

In [41]:
q1 = orders['sales_amount'].quantile(0.25)
print(q1)

113252.0


2. 여러 분위수를 한 번에 계산  
분위수 위치를 리스트로 전달하면 여러 분위수를 한 번에 계산할 수 있다.

In [42]:
result = orders['sales_amount'].quantile([0.25, 0.50, 0.75])
print(result.to_string())

0.25    113252.0
0.50    277067.5
0.75    571125.0


3. 여러 칼럼의 분위수  
컬럼도 여러 개를 동시에 지정할 수 있다. 여러 컬럼을 선택한 데이터프레임에 quantile()을 적용해본다

In [46]:
cols = ['sales_amount', 'unit_price', 'qty']
result = orders[cols].quantile([0.25, 0.50, 0.75])
print(result.to_string())

      sales_amount  unit_price  qty
0.25      113252.0     34525.0  3.0
0.50      277067.5     63150.0  6.0
0.75      571125.0     92000.0  9.0


분위수를 기준으로 필터링  
분위수는 계산으로 끝나지 않고 필터링 기준으로 이어지는 경우가 많다. 3사분위수를 구한 뒤 그 이상인 주문만 골라내 보자!


In [48]:
q3 = orders['sales_amount'].quantile(0.75)
high_orders = orders[orders['sales_amount'] >= q3]

print(q3)
print(len(high_orders))
print(high_orders[['order_id','category', 'sales_amount']].head().to_string(index=False))

571125.0
50
order_id category  sales_amount
  O00002  kitchen       1008700
  O00010     book       1249600
  O00012  digital        737100
  O00013     book       1248390
  O00026  kitchen        951900


#### 11. 반올림, 버림, 올림  
작업형 문제에서는 계산 결과를 문제에서 요구한 자릿수에 맞게 반올림하거나 정수로 바꾸어 제출해야하는 경우가 많다.  
평균, 비율과 같이 소수점이 생기는 값은 마지막 출력 전에 처리 기준을 확인해야 한다.  
반올림은 round(), 내림과 올림은 넘파이의 floor(), ceil()을 사용한다. 양수 데이터에서는 floor()를 소수점 이하 버림처럼  
사용할 수 있다. 같은 숫자라도 처리 방식에 따라 답이 달라지므로 문제 문장의 표현을 정확히 구분해야 한다.

In [49]:
# 금액 단위 변환  
orders['amount_만원'] = orders['sales_amount'] / 10000
print(orders[['order_id', 'sales_amount', 'amount_만원']].head().to_string(index=False))

order_id  sales_amount  amount_만원
  O00001        304095    30.4095
  O00002       1008700   100.8700
  O00003        116982    11.6982
  O00004        302500    30.2500
  O00005         50000     5.0000


#### 반올림  
round()에 자릿수를 전달하면 그 자리까지 반올림한다. 소수 첫째 자리로 반올림함

In [50]:
orders['반올림_1자리'] = orders['amount_만원'].round(1)
print(orders[['order_id', 'amount_만원', '반올림_1자리']].head().to_string(index=False))

order_id  amount_만원  반올림_1자리
  O00001    30.4095     30.4
  O00002   100.8700    100.9
  O00003    11.6982     11.7
  O00004    30.2500     30.2
  O00005     5.0000      5.0


In [51]:
# 정수 자리 반올림
orders['반올림_0자리'] = orders['amount_만원'].round(0)
print(orders[['order_id', 'amount_만원', '반올림_0자리']].head().to_string(index=False))

order_id  amount_만원  반올림_0자리
  O00001    30.4095     30.0
  O00002   100.8700    101.0
  O00003    11.6982     12.0
  O00004    30.2500     30.0
  O00005     5.0000      5.0


버림과 올림  
버림과 올림은 판다스가 아니라 넘파이를 이용함. 결과를 정수형으로 만들기 위해 astype(int)를 함께 적용한다

In [52]:
orders['버림'] = np.floor(orders['amount_만원'].astype(int))
orders['올림'] = np.ceil(orders['amount_만원'].astype(int))

print(orders[['order_id', 'amount_만원', '버림', '올림']].head().to_string(index=False))

order_id  amount_만원    버림    올림
  O00001    30.4095  30.0  30.0
  O00002   100.8700 100.0 100.0
  O00003    11.6982  11.0  11.0
  O00004    30.2500  30.0  30.0
  O00005     5.0000   5.0   5.0


In [54]:
# 계산결과 하나 처리하기
avg = orders['sales_amount'].mean()

print(avg)
print(round(avg, 2))
print(round(avg))
print(int(np.floor(avg)))
print(int(np.ceil(avg)))

375104.66
375104.66
375105
375104
375105


In [55]:
category_sales = orders.groupby('category')['sales_amount'].sum()
print(category_sales.idxmax())

office


In [57]:
num = orders[['qty', 'unit_price', 'discount_rate', 'sales_amount']]
corr = num.corr()

ans = corr['sales_amount'].drop('sales_amount').abs().idxmax()
print(ans)

unit_price
